In [ ]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.0/37.0 MB 43.3 MB/s eta 0:00:00


In [ ]:
from rdkit import Chem
from rdkit.Chem import FilterCatalog
from rdkit.Chem.FilterCatalog import FilterCatalogParams

In [ ]:
pains_params = FilterCatalogParams()
pains_params.AddCatalog(FilterCatalogParams.FilterCatalogs.PAINS)
PAINS_catalog = FilterCatalog.FilterCatalog(pains_params)

def calc_pains(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    matches = PAINS_catalog.GetMatches(mol)
    if matches:
        return True, [match.GetDescription() for match in matches]
    return False, []

In [ ]:
brenk_params = FilterCatalogParams()
brenk_params.AddCatalog(FilterCatalogParams.FilterCatalogs.BRENK)
BRENK_catalog = FilterCatalog.FilterCatalog(brenk_params)

def calc_brenk(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    matches = BRENK_catalog.GetMatches(mol)
    if matches:
        return True, [match.GetDescription() for match in matches]
    return False, []

In [ ]:
import pandas as pd

csv_file_path = 'smiles.csv'
df = pd.read_csv(csv_file_path)

print(df.head())

            Name                                    SMILES           Status  \
0  ACETAMINOPHEN                        CC(=O)Nc1ccc(O)cc1         good bro   
1     PHENACETIN                      CCOc1ccc(NC(C)=O)cc1          bad bro   
2    S-IBUPROFEN           CC(C)Cc1ccc([C@@H](C)C(=O)O)cc1  more active bro   
3    R-IBUPROFEN            CC(C)Cc1ccc([C@H](C)C(=O)O)cc1  less active bro   
4  R-THALIDOMIDE  O=C1CC[C@@H](N2C(=O)c3ccccc3C2=O)C(=O)N1         good bro   

                                  Comment  \
0   имеет -OH группу у бензольного кольца   
1  имеет -EtO группу у бензольного кольца   
2                          s-стереоизомер   
3                          r-стереоизомер   
4                            r-энантиомер   

                                         Explanation  Столбец 1  
0          не имеет побочек при нормальной дозировке        NaN  
1  является канцерогенном и обладает нефротоксичн...        NaN  
2      обладает нужными терапевтическими свойствами     

In [ ]:
def apply_filters(row):
    smiles = row['SMILES']
    pains_res = calc_pains(smiles)
    brenk_res = calc_brenk(smiles)

    # Выпаковываем результаты (bool, list)
    row['BRENK'] = brenk_res[0] if brenk_res else None
    row['BRENK_structs'] = ", ".join(brenk_res[1]) if brenk_res and brenk_res[1] else ""

    row['PAINS'] = pains_res[0] if pains_res else None
    row['PAINS_structs'] = ", ".join(pains_res[1]) if pains_res and pains_res[1] else ""

    return row

df = df.apply(apply_filters, axis=1)
print(df[['SMILES', 'PAINS', 'PAINS_structs', 'BRENK', 'BRENK_structs']].head())

                                     SMILES  PAINS PAINS_structs  BRENK  \
0                        CC(=O)Nc1ccc(O)cc1  False                 True   
1                      CCOc1ccc(NC(C)=O)cc1  False                False   
2           CC(C)Cc1ccc([C@@H](C)C(=O)O)cc1  False                False   
3            CC(C)Cc1ccc([C@H](C)C(=O)O)cc1  False                False   
4  O=C1CC[C@@H](N2C(=O)c3ccccc3C2=O)C(=O)N1  False                 True   

  BRENK_structs  
0  hydroquinone  
1                
2                
3                
4   phthalimide  


In [ ]:
df.to_csv("results.csv")

## Ручная проверка

In [ ]:
smiles = '[N-]=[N+]=NC1(CO)OC(n2ccc(N)nc2=O)C(F)C1O'
print("BRENK:", calc_brenk(smiles))
print("PAINS:", calc_pains(smiles))

BRENK: (True, ['Azido_group', 'diazo_group', 'quaternary_nitrogen_3'])
PAINS: (True, ['azo_A(324)'])
